# Filtrado 18 - Degree of automation
Consiste en sacar las dwas asociadas a ocupaciones con su degree of automation

## SQL

In [ ]:
# -*- coding: utf-8 -*-
from pathlib import Path

# Lista tal como la diste
ocupations = [
    'Industrial Production Managers','Quality Control Systems Managers','Supply Chain Managers',
    'Human Resources Managers','Logistics Engineers','Bioengineers and Biomedical Engineers',
    'Chemical Engineers','Validation Engineers','Manufacturing Engineers','Mechanical Engineers',
    'Automotive Engineers','Mechatronics Engineers','Robotics Engineers','Robotics Technicians',
    'Industrial Engineering Technologists and Technicians',
    'Mechanical Engineering Technologists and Technicians',
    'Biochemists and Biophysicists','Microbiologists','Chemists'
]


In [ ]:

# Carpeta de salida (cámbiala si quieres)
OUT_DIR = Path("filtrado18_degree_of_automation")
OUT_DIR.mkdir(parents=True, exist_ok=True)

TEMPLATE = """USE onet;

SELECT DISTINCT
    o.onetsoc_code,
    o.title,
    td.dwa_id,
    dwaref.dwa_title,
    w.data_value AS degree_of_automation
FROM tasks_to_dwas td
JOIN occupation_data o      ON td.onetsoc_code = o.onetsoc_code
JOIN dwa_reference dwaref   ON td.dwa_id      = dwaref.dwa_id
JOIN work_context w         ON o.onetsoc_code = w.onetsoc_code
JOIN content_model_reference cmr ON w.element_id = cmr.element_id
WHERE o.title = '{title_escaped}'
  AND w.scale_id = 'CX'
  AND cmr.element_name = 'Degree of Automation'
ORDER BY dwaref.dwa_title;

"""


In [ ]:

for title in ocupations:
    # title=title.lower()
    # Escapar comillas simples para SQL (por si en el futuro aparece alguna)
    title_escaped = title.replace("'", "''")
    # Nombre de archivo: espacios -> _
    filename = title.replace(" ", "_") + ".sql"
    sql_text = TEMPLATE.format(title_escaped=title_escaped)
    (OUT_DIR / filename).write_text(sql_text, encoding="utf-8")

print(f"Generados {len(ocupations)} archivos .sql en: {OUT_DIR.resolve()}")


## Agrupacion de todos los datasets

In [22]:
import os
import pandas as pd

path = "/Users/jpardo/Desktop/Proyectos/AI4LABOUR/mejoras/eerr/filtrado18_degree_of_automation"

# Listar solo CSV
archivos = [i for i in os.listdir(path) if i.endswith('.csv')]

dfs = []  # lista para acumular dataframes

for i in archivos:
    path_archivo = os.path.join(path, i)
    df_temp = pd.read_csv(path_archivo)
    df_temp["archivo_origen"] = i  # opcional: para saber de qué archivo viene cada fila
    dfs.append(df_temp)

# Unir todos los DataFrames
df_final = pd.concat(dfs, ignore_index=True)

print(f"Se han cargado {len(df_final)} filas de {len(archivos)} archivos.")


Se han cargado 422 filas de 19 archivos.


In [23]:
display(df_final)

,onetsoc_code,title,dwa_id,dwa_title,degree_of_automation,archivo_origen
0,17-2141.02,Automotive Engineers,4.A.3.a.2.I09.D04,Calibrate scientific or technical equipment.,2.24,Automotive_Engineers.csv
1,17-2141.02,Automotive Engineers,4.A.2.a.4.I12.D01,Conduct quantitative failure analyses of opera...,2.24,Automotive_Engineers.csv
2,17-2141.02,Automotive Engineers,4.A.4.a.2.I11.D07,"Coordinate activities with suppliers, contract...",2.24,Automotive_Engineers.csv
3,17-2141.02,Automotive Engineers,4.A.2.b.2.I21.D01,Create models of engineering designs or methods.,2.24,Automotive_Engineers.csv
4,17-2141.02,Automotive Engineers,4.A.2.b.2.I14.D06,Design control systems for mechanical or other...,2.24,Automotive_Engineers.csv
...,...,...,...,...,...,...
417,11-3051.01,Quality Control Systems Managers,4.A.3.b.6.I15.D06,Prepare operational progress or status reports.,2.25,Quality_Control_Systems_Managers.csv
418,11-3051.01,Quality Control Systems Managers,4.A.4.b.6.I05.D07,Recommend organizational process or policy cha...,2.25,Quality_Control_Systems_Managers.csv
419,11-3051.01,Quality Control Systems Managers,4.A.1.a.1.I02.D15,Review details of technical drawings or specif...,2.25,Quality_Control_Systems_Managers.csv
420,11-3051.01,Quality Control Systems Managers,4.A.2.a.3.I01.D04,Review documents or materials for compliance w...,2.25,Quality_Control_Systems_Managers.csv


In [24]:
!pip install openpyxl


[notice] A new release of pip is available: 25.1.1 -> 25.2
[notice] To update, run: pip install --upgrade pip


In [25]:
# Cargar el Excel
ruta_excel = "/Users/jpardo/Desktop/Proyectos/AI4LABOUR/mejoras/eerr/Jobs List ICBE.xlsx"
df_excel = pd.read_excel(ruta_excel, sheet_name="ICBE")
# Pasar nombres de columnas a minúsculas
df_excel.columns = df_excel.columns.str.lower()


display(df_excel)

,o.net.soc.code,title,icbe,description
0,11-3051.00,Industrial Production Managers,Primary,"Plan, direct, or coordinate the work activitie..."
1,11-3051.01,Quality Control Systems Managers,Secondary,"Plan, direct, or coordinate quality assurance ..."
2,11-3071.04,Supply Chain Managers,Secondary,"Direct or coordinate production, purchasing, w..."
3,11-3121.00,Human Resources Managers,Secondary,"Plan, direct, or coordinate human resources ac..."
4,13-1081.01,Logistics Engineers,Secondary,Design or analyze operational solutions for pr...
5,17-2031.00,Bioengineers and Biomedical Engineers,NaN,"Apply knowledge of engineering, biology, chemi..."
6,17-2041.00,Chemical Engineers,Primary,Design chemical plant equipment and devise pro...
7,17-2112.02,Validation Engineers,NaN,Design or plan protocols for equipment or proc...
8,17-2112.03,Manufacturing Engineers,Primary,"Design, integrate, or improve manufacturing sy..."
9,17-2141.00,Mechanical Engineers,Primary,Perform engineering duties in planning and des...


In [26]:
# Unir DataFrames
# Aquí debes definir la columna clave común en ambos DF, por ejemplo 'dwa_id'
df_merged = pd.merge(df_final, df_excel, on="title", how="left")

print(f"DataFrame final tiene {df_merged.shape[0]} filas y {df_merged.shape[1]} columnas.")

DataFrame final tiene 422 filas y 9 columnas.


In [27]:
display(df_merged)

,onetsoc_code,title,dwa_id,dwa_title,degree_of_automation,archivo_origen,o.net.soc.code,icbe,description
0,17-2141.02,Automotive Engineers,4.A.3.a.2.I09.D04,Calibrate scientific or technical equipment.,2.24,Automotive_Engineers.csv,17-2141.02,NaN,Develop new or improved designs for vehicle st...
1,17-2141.02,Automotive Engineers,4.A.2.a.4.I12.D01,Conduct quantitative failure analyses of opera...,2.24,Automotive_Engineers.csv,17-2141.02,NaN,Develop new or improved designs for vehicle st...
2,17-2141.02,Automotive Engineers,4.A.4.a.2.I11.D07,"Coordinate activities with suppliers, contract...",2.24,Automotive_Engineers.csv,17-2141.02,NaN,Develop new or improved designs for vehicle st...
3,17-2141.02,Automotive Engineers,4.A.2.b.2.I21.D01,Create models of engineering designs or methods.,2.24,Automotive_Engineers.csv,17-2141.02,NaN,Develop new or improved designs for vehicle st...
4,17-2141.02,Automotive Engineers,4.A.2.b.2.I14.D06,Design control systems for mechanical or other...,2.24,Automotive_Engineers.csv,17-2141.02,NaN,Develop new or improved designs for vehicle st...
...,...,...,...,...,...,...,...,...,...
417,11-3051.01,Quality Control Systems Managers,4.A.3.b.6.I15.D06,Prepare operational progress or status reports.,2.25,Quality_Control_Systems_Managers.csv,11-3051.01,Secondary,"Plan, direct, or coordinate quality assurance ..."
418,11-3051.01,Quality Control Systems Managers,4.A.4.b.6.I05.D07,Recommend organizational process or policy cha...,2.25,Quality_Control_Systems_Managers.csv,11-3051.01,Secondary,"Plan, direct, or coordinate quality assurance ..."
419,11-3051.01,Quality Control Systems Managers,4.A.1.a.1.I02.D15,Review details of technical drawings or specif...,2.25,Quality_Control_Systems_Managers.csv,11-3051.01,Secondary,"Plan, direct, or coordinate quality assurance ..."
420,11-3051.01,Quality Control Systems Managers,4.A.2.a.3.I01.D04,Review documents or materials for compliance w...,2.25,Quality_Control_Systems_Managers.csv,11-3051.01,Secondary,"Plan, direct, or coordinate quality assurance ..."


### Ocuppations

In [28]:
from pprint import pprint
occupations = df_excel['title'].tolist()
pprint(occupations)

['Industrial Production Managers',
 'Quality Control Systems Managers',
 'Supply Chain Managers',
 'Human Resources Managers',
 'Logistics Engineers',
 'Bioengineers and Biomedical Engineers',
 'Chemical Engineers',
 'Validation Engineers',
 'Manufacturing Engineers',
 'Mechanical Engineers',
 'Automotive Engineers',
 'Mechatronics Engineers',
 'Robotics Engineers',
 'Robotics Technicians',
 'Industrial Engineering Technologists and Technicians',
 'Mechanical Engineering Technologists and Technicians',
 'Biochemists and Biophysicists',
 'Microbiologists',
 'Chemists']


In [29]:
print(len(occupations))

19


### DWAs

In [30]:
conteo = pd.DataFrame(df_final["dwa_title"].value_counts())

display(conteo)


,count
dwa_title,
Estimate operational costs.,8
Design industrial processing systems.,7
"Recommend technical design or process changes to improve efficiency, quality, or performance.",7
Maintain operational records or records systems.,6
Train personnel on proper operational procedures.,6
...,...
Coordinate with external parties to exchange information.,1
Repair electronic equipment.,1
"Maintain inventories of materials, equipment, or products.",1


In [31]:
num_compartidos = conteo[conteo['count'] > 1]
display(num_compartidos)

,count
dwa_title,
Estimate operational costs.,8
Design industrial processing systems.,7
"Recommend technical design or process changes to improve efficiency, quality, or performance.",7
Maintain operational records or records systems.,6
Train personnel on proper operational procedures.,6
...,...
"Investigate system, equipment, or product failures.",2
Research diseases or parasites.,2
Design electronic or computer equipment or instrumentation.,2


In [32]:
221-100

121

Hay 121 dwa que se comparten entre las 19 occupations. 

In [33]:
dwa = df_final['dwa_title'].unique().tolist()
pprint(dwa)

['Calibrate scientific or technical equipment.',
 'Conduct quantitative failure analyses of operational data.',
 'Coordinate activities with suppliers, contractors, clients, or other '
 'departments.',
 'Create models of engineering designs or methods.',
 'Design control systems for mechanical or other equipment.',
 'Design electromechanical equipment or systems.',
 'Design energy-efficient vehicles or vehicle components.',
 'Determine design criteria or specifications.',
 'Determine operational criteria or specifications.',
 'Develop technical methods or processes.',
 'Devise research or testing protocols.',
 'Direct design or development activities.',
 'Estimate operational costs.',
 'Evaluate characteristics of equipment or systems.',
 'Evaluate technical data to determine effect on designs or plans.',
 'Implement design or process improvements.',
 'Maintain operational records or records systems.',
 'Prepare operational reports.',
 'Prepare technical reports for internal use.',
 'P

In [34]:
print(len(dwa))

221


## Filtrado 18 - degree of automation numeric

In [35]:
# Eliminar columnas no deseadas
occupations2dwa = df_final[["title", "dwa_title", "degree_of_automation"]]

display(occupations2dwa)

,title,dwa_title,degree_of_automation
0,Automotive Engineers,Calibrate scientific or technical equipment.,2.24
1,Automotive Engineers,Conduct quantitative failure analyses of opera...,2.24
2,Automotive Engineers,"Coordinate activities with suppliers, contract...",2.24
3,Automotive Engineers,Create models of engineering designs or methods.,2.24
4,Automotive Engineers,Design control systems for mechanical or other...,2.24
...,...,...,...
417,Quality Control Systems Managers,Prepare operational progress or status reports.,2.25
418,Quality Control Systems Managers,Recommend organizational process or policy cha...,2.25
419,Quality Control Systems Managers,Review details of technical drawings or specif...,2.25
420,Quality Control Systems Managers,Review documents or materials for compliance w...,2.25


In [36]:
degree_avg_dwa = (
    occupations2dwa
    .groupby("dwa_title")["degree_of_automation"]
    .mean()
    .reset_index()
)

display(degree_avg_dwa)


,dwa_title,degree_of_automation
0,Administer compensation or benefits programs.,2.080000
1,Administer standardized physical or psychologi...,2.080000
2,Advise customers on technical or procedural is...,2.250000
3,Advise customers on the use of products or ser...,2.290000
4,Advise others on career or personal development.,2.080000
...,...,...
216,Test products for functionality or quality.,2.170000
217,Test quality of materials or finished products.,2.020000
218,Train personnel on proper operational procedures.,2.303333
219,Update technical knowledge.,2.165000


In [37]:
import re, unicodedata

# --- Regex precompiles ---
RE_ZERO_WIDTH   = re.compile(r'[\u200B-\u200F\u202A-\u202E\u2060\uFEFF]')
RE_SPACES       = re.compile(r'\s+')
RE_TRAILING_DOT = re.compile(r'[.\s]+$')

# Clase de comillas (ASCII + tipográficas + angulares)
QUOTE_UTF8_CHARS = '"\'`“”„‟‘’‚‛«»‹›'
RE_EDGE_QUOTES   = re.compile(rf'^[{re.escape(QUOTE_UTF8_CHARS)}\s]+|[{re.escape(QUOTE_UTF8_CHARS)}\s]+$')

# Mojibake -> UTF-8/ASCII
MOJIBAKE_MAP = {
    "â¢": "•",
    "â": "—",
    "â": "–",
    "â¦": "…",
    "â": "'",
    "â": "'",
    "â": '"',
    "â": '"',
    "â": "",
}

# Enumeraciones iniciales (NO metas comillas aquí)
LEADING_ENUM_PATTERNS = [
    re.compile(r'^\s*\d+\s*\t+\s*'),               # "12\t..."
    re.compile(r'^\s*\d+\s*[\.\)]\s*'),            # "1." / "2)"
    re.compile(r'^\s*[\(\[]\s*\d+\s*[\)\]]\s*'),   # "(1)" / "[2]"
    re.compile(r'^\s*\d+\s*[-–—]\s+'),             # "1 - " / "1 – " / "1 — "
    re.compile(r'^\s*[A-Za-z]\s*[\.\)]\s+'),       # "a) " / "B. "
    re.compile(r'^\s*[ivxlcdmIVXLCDM]+\s*[\.\)]\s+'),  # "IV. " / "i) "
    re.compile(r'^\s*[•\-\*\u2219·–—]\s*\t+\s*'),  # bullets con tab
    re.compile(r'^\s*[•\-\*\u2219·–—]\s+'),        # bullets con espacio
]

def _strip_edge_quotes_loop(s: str) -> str:
    """Recorta comillas/espacios en bordes de forma repetida (izq./der.)."""
    while True:
        new_s = RE_EDGE_QUOTES.sub('', s).strip()
        if new_s == s:
            return s
        s = new_s

def clean_skill(text: str) -> str:
    """
    Devuelve la skill limpia en minúsculas:
      - quita enumeraciones iniciales,
      - recorta comillas de borde (ASCII/UTF-8),
      - corrige mojibake, elimina zero-width/bidi/BOM,
      - quita punto(s) finales,
      - conserva paréntesis internos.
    """
    if text is None:
        return ""
    s = unicodedata.normalize("NFKC", str(text))

    # Mojibake -> UTF8/ASCII
    for bad, good in MOJIBAKE_MAP.items():
        if bad in s:
            s = s.replace(bad, good)

    # Invisibles
    s = RE_ZERO_WIDTH.sub("", s)

    # Enumeraciones al inicio (pelar en capas)
    prev = None
    while s != prev:
        prev = s
        for pat in LEADING_ENUM_PATTERNS:
            s = pat.sub("", s, count=1)

    # Normaliza espacios
    s = RE_SPACES.sub(" ", s).strip()

    # 1ª pasada: recortar comillas de borde
    s = _strip_edge_quotes_loop(s)

    # Quitar punto(s) finales
    s = RE_TRAILING_DOT.sub("", s).strip()

    # 2ª pasada: recortar comillas de borde (para casos como ”.)
    s = _strip_edge_quotes_loop(s)

    # Minúsculas
    return s.lower().strip()

def _t(inp, expected):
    got = clean_skill(inp)
    assert got == expected, f"\nINPUT : {repr(inp)}\nGOT   : {repr(got)}\nEXPECT: {repr(expected)}"

# Comillas (ASCII, tipográficas, angulares) + punto final
_t('"Python Programming"', 'python programming')
_t('“Python Programming”.', 'python programming')
_t("‘Skill’", 'skill')
_t("«Skill»", 'skill')
_t("‹Skill›", 'skill')
_t("`Skill`", 'skill')
_t("' Skill '", 'skill')

# Punto final en skills largas
_t("Long descriptive skill.", "long descriptive skill")
_t("C++.", "c++")

# Mayúsculas/minúsculas
_t("JAVA", "java")
_t("Title Case Mixed", "title case mixed")

# Enumeraciones (número + tab / . / ) / (1) / [2] / 1 - / letra) / romanos / bullets
_t("12\tSystems Design", "systems design")
_t("1.\tSkill", "skill")
_t("(1)\tSkill", "skill")
_t("[2] Skill", "skill")
_t("1 - Skill", "skill")
_t("a)\tSkill", "skill")
_t("IV.\tSkill", "skill")
_t("•\tLeadership", "leadership")
_t("-\tProject Management", "project management")
_t("—\tSystems", "systems")
_t("–\tSystems", "systems")
_t("• bullet style skill", "bullet style skill")
_t("·\tAnother Skill", "another skill")

# Mojibake en medio del texto + zero-width
_t("4Aâs Marketing", "4a's marketing")
_t("Hâow to Design", "how to design")

# Paréntesis preservados
_t("Design (CAD)", "design (cad)")
_t("A/B Testing (advanced)", "a/b testing (advanced)")


In [38]:
temp=degree_avg_dwa['dwa_title'].apply(clean_skill)


display(temp)

0           administer compensation or benefits programs
1      administer standardized physical or psychologi...
2      advise customers on technical or procedural is...
3      advise customers on the use of products or ser...
4        advise others on career or personal development
                             ...                        
216           test products for functionality or quality
217       test quality of materials or finished products
218     train personnel on proper operational procedures
219                           update technical knowledge
220                                write grant proposals
Name: dwa_title, Length: 221, dtype: object

In [39]:
conteo = pd.DataFrame(temp.value_counts())

display(conteo)

,count
dwa_title,
administer compensation or benefits programs,1
monitor environmental impacts of production or development activities,1
maintain laboratory or technical equipment,1
maintain operational records or records systems,1
maintain personnel records,1
...,...
develop specifications for new products or processes,1
develop sustainable business strategies or practices,1
develop sustainable organizational policies or practices,1


In [40]:
for i in temp:
    print(i) 


administer compensation or benefits programs
administer standardized physical or psychological tests
advise customers on technical or procedural issues
advise customers on the use of products or services
advise others on career or personal development
advise others on legal or regulatory compliance matters
advise others on logistics topics
advise others regarding green practices or environmental concerns
analyze biological samples
analyze chemical compounds or substances
analyze costs and benefits of proposed designs or projects
analyze data to assess operational or project effectiveness
analyze data to inform operational decisions or activities
analyze data to inform personnel decisions
analyze design or requirements information for mechanical equipment or systems
analyze environmental regulations to ensure organizational compliance
analyze green technology design requirements
analyze jobs using observation, survey, or interview techniques
analyze logistics processes
analyze operation

In [41]:
degree_avg_dwa2 = degree_avg_dwa.copy()
degree_avg_dwa2['dwa_title'] = temp

display(degree_avg_dwa2)

,dwa_title,degree_of_automation
0,administer compensation or benefits programs,2.080000
1,administer standardized physical or psychologi...,2.080000
2,advise customers on technical or procedural is...,2.250000
3,advise customers on the use of products or ser...,2.290000
4,advise others on career or personal development,2.080000
...,...,...
216,test products for functionality or quality,2.170000
217,test quality of materials or finished products,2.020000
218,train personnel on proper operational procedures,2.303333
219,update technical knowledge,2.165000


In [43]:
print(path)

/Users/jpardo/Desktop/Proyectos/AI4LABOUR/mejoras/eerr/filtrado18_degree_of_automation


In [45]:
import csv
from pathlib import Path

formato =[".csv", ".tsv"] 

OUT_DIR2 = Path(path+"/results")
OUT_DIR2.mkdir(parents=True, exist_ok=True)

for i in formato:
    degree_avg_dwa2.to_csv(f"{OUT_DIR2}/filtrado18_results_degree_num{i}", index=False, encoding="utf-8",
                        quoting=csv.QUOTE_MINIMAL, quotechar='"', escapechar='\\')


## Filtrado 18 - degree of automation categoric

In [46]:
import pandas as pd

# Asegurar que la columna es numérica
degree_avg_dwa["degree_of_automation"] = pd.to_numeric(degree_avg_dwa["degree_of_automation"], errors="coerce")

# Crear columna de cuartiles (2 grupos)
degree_avg_dwa["automation_quartile"] = pd.qcut(degree_avg_dwa["degree_of_automation"], q=2, labels=["Bajo", "Alto"])

display(degree_avg_dwa[["dwa_title", "degree_of_automation", "automation_quartile"]])


,dwa_title,degree_of_automation,automation_quartile
0,Administer compensation or benefits programs.,2.080000,Bajo
1,Administer standardized physical or psychologi...,2.080000,Bajo
2,Advise customers on technical or procedural is...,2.250000,Alto
3,Advise customers on the use of products or ser...,2.290000,Alto
4,Advise others on career or personal development.,2.080000,Bajo
...,...,...,...
216,Test products for functionality or quality.,2.170000,Bajo
217,Test quality of materials or finished products.,2.020000,Bajo
218,Train personnel on proper operational procedures.,2.303333,Alto
219,Update technical knowledge.,2.165000,Bajo


In [47]:
conteo2 = pd.DataFrame(degree_avg_dwa["automation_quartile"].value_counts())

display(conteo2)

,count
automation_quartile,
Bajo,111
Alto,110


In [48]:
degree_avg_dwa_cat = degree_avg_dwa[["dwa_title", "automation_quartile"]]
display(degree_avg_dwa_cat)

,dwa_title,automation_quartile
0,Administer compensation or benefits programs.,Bajo
1,Administer standardized physical or psychologi...,Bajo
2,Advise customers on technical or procedural is...,Alto
3,Advise customers on the use of products or ser...,Alto
4,Advise others on career or personal development.,Bajo
...,...,...
216,Test products for functionality or quality.,Bajo
217,Test quality of materials or finished products.,Bajo
218,Train personnel on proper operational procedures.,Alto
219,Update technical knowledge.,Bajo


In [49]:
temp=degree_avg_dwa_cat['dwa_title'].apply(clean_skill)


display(temp)

0           administer compensation or benefits programs
1      administer standardized physical or psychologi...
2      advise customers on technical or procedural is...
3      advise customers on the use of products or ser...
4        advise others on career or personal development
                             ...                        
216           test products for functionality or quality
217       test quality of materials or finished products
218     train personnel on proper operational procedures
219                           update technical knowledge
220                                write grant proposals
Name: dwa_title, Length: 221, dtype: object

In [50]:
conteo = pd.DataFrame(temp.value_counts())

display(conteo)

,count
dwa_title,
administer compensation or benefits programs,1
monitor environmental impacts of production or development activities,1
maintain laboratory or technical equipment,1
maintain operational records or records systems,1
maintain personnel records,1
...,...
develop specifications for new products or processes,1
develop sustainable business strategies or practices,1
develop sustainable organizational policies or practices,1


In [51]:
for i in temp:
    print(i) 


administer compensation or benefits programs
administer standardized physical or psychological tests
advise customers on technical or procedural issues
advise customers on the use of products or services
advise others on career or personal development
advise others on legal or regulatory compliance matters
advise others on logistics topics
advise others regarding green practices or environmental concerns
analyze biological samples
analyze chemical compounds or substances
analyze costs and benefits of proposed designs or projects
analyze data to assess operational or project effectiveness
analyze data to inform operational decisions or activities
analyze data to inform personnel decisions
analyze design or requirements information for mechanical equipment or systems
analyze environmental regulations to ensure organizational compliance
analyze green technology design requirements
analyze jobs using observation, survey, or interview techniques
analyze logistics processes
analyze operation

In [52]:
degree_avg_dwa_cat2 = degree_avg_dwa_cat.copy()
degree_avg_dwa_cat2['dwa_title'] = temp

display(degree_avg_dwa_cat2)

,dwa_title,automation_quartile
0,administer compensation or benefits programs,Bajo
1,administer standardized physical or psychologi...,Bajo
2,advise customers on technical or procedural is...,Alto
3,advise customers on the use of products or ser...,Alto
4,advise others on career or personal development,Bajo
...,...,...
216,test products for functionality or quality,Bajo
217,test quality of materials or finished products,Bajo
218,train personnel on proper operational procedures,Alto
219,update technical knowledge,Bajo


In [53]:
degree_avg_dwa_cat2.shape

(221, 2)

In [54]:
import csv

for i in formato:
    degree_avg_dwa_cat2.to_csv(f"{OUT_DIR2}/filtrado18_results_degree_cat{i}", index=False, encoding="utf-8",
                        quoting=csv.QUOTE_MINIMAL, quotechar='"', escapechar='\\')